In [1]:
from pathlib import Path
file_path = Path('./data/federalist_papers.txt')

In [2]:
# I should probably write a code that checks if smth is smth 
# import requests
# r = requests.get('https://www.gutenberg.org/cache/epub/1404/pg1404.txt')
# with open(file_path , 'wb') as f:
#     f.write(r.content)

In [3]:
import spacy 
nlp = spacy.load('en_core_web_md')
nlp.add_pipe('sentencizer')

In [4]:
f = open(file_path, 'r', encoding='utf-8')

collection_paper = []
federalist_paper = []

for f_line in f:
    if f_line.startswith('*** END OF THE PROJECT GUTENBERG'):
        break
    if f_line.startswith('FEDERALIST No.'):
        federalist_paper = [] 
        collection_paper.append(federalist_paper)
    federalist_paper.append(f_line)


print(collection_paper[0])
print(len(collection_paper))
f.close()

['FEDERALIST No. 1\n', '\n', 'General Introduction\n', '\n', 'For the Independent Journal. Saturday, October 27, 1787\n', '\n', '\n', 'HAMILTON\n', '\n', 'To the People of the State of New York:\n', '\n', 'AFTER an unequivocal experience of the inefficacy of the subsisting\n', 'federal government, you are called upon to deliberate on a new\n', 'Constitution for the United States of America. The subject speaks its\n', 'own importance; comprehending in its consequences nothing less than the\n', 'existence of the UNION, the safety and welfare of the parts of which it\n', 'is composed, the fate of an empire in many respects the most interesting\n', 'in the world. It has been frequently remarked that it seems to have been\n', 'reserved to the people of this country, by their conduct and example,\n', 'to decide the important question, whether societies of men are really\n', 'capable or not of establishing good government from reflection and\n', 'choice, or whether they are forever destined t

In [5]:
import re 

collection_paper = [[line.strip() for line in paper if line.strip()] for paper in collection_paper]

author_list = ('HAMILTON', 'JAY', 'MADISON')
collection_dict = []

for paper in collection_paper:
    paper_dict = {'paper_no': 0, 'supposed_author': '', 'content': [], 'reference': []} 

    state = 'CONTENT'
    for line in paper:
        if line.startswith('FEDERALIST No'):
            paper_dict['paper_no'] = int(re.search(r'\d+', line).group())
            continue
        if line.startswith(author_list):
            paper_dict['supposed_author'] = line
            continue
        if line == 'PUBLIUS':
            state = 'REFERENCE'
            continue
        if line.startswith('To the People of the State'):
            paper_dict['content'] = []
            continue
        if state == 'CONTENT':
            cleaned = re.sub(r'\(\d+\)', '', line)
            paper_dict['content'].append(cleaned)
        else:
            paper_dict['reference'].append(line)
    paper_dict['content'] = ' '.join(paper_dict['content'])
    collection_dict.append(paper_dict)

print(collection_dict[0])

{'paper_no': 1, 'supposed_author': 'HAMILTON', 'content': 'AFTER an unequivocal experience of the inefficacy of the subsisting federal government, you are called upon to deliberate on a new Constitution for the United States of America. The subject speaks its own importance; comprehending in its consequences nothing less than the existence of the UNION, the safety and welfare of the parts of which it is composed, the fate of an empire in many respects the most interesting in the world. It has been frequently remarked that it seems to have been reserved to the people of this country, by their conduct and example, to decide the important question, whether societies of men are really capable or not of establishing good government from reflection and choice, or whether they are forever destined to depend for their political constitutions on accident and force. If there be any truth in the remark, the crisis at which we are arrived may with propriety be regarded as the era in which that dec

In [6]:
for paper in collection_dict:
    print(paper['paper_no'], paper['supposed_author'])

1 HAMILTON
2 JAY
3 JAY
4 JAY
5 JAY
6 HAMILTON
7 HAMILTON
8 HAMILTON
9 HAMILTON
10 MADISON
11 HAMILTON
12 HAMILTON
13 HAMILTON
14 MADISON
15 HAMILTON
16 HAMILTON
17 HAMILTON
18 MADISON, with HAMILTON
19 MADISON, with HAMILTON
20 MADISON, with HAMILTON
21 HAMILTON
22 HAMILTON
23 HAMILTON
24 HAMILTON
25 HAMILTON
26 HAMILTON
27 HAMILTON
28 HAMILTON
29 HAMILTON
30 HAMILTON
31 HAMILTON
32 HAMILTON
33 HAMILTON
34 HAMILTON
35 HAMILTON
36 HAMILTON
37 MADISON
38 MADISON
39 MADISON
40 MADISON
41 MADISON
42 MADISON
43 MADISON
44 MADISON
45 MADISON
46 MADISON
47 MADISON
48 MADISON
49 MADISON
50 MADISON
51 MADISON
52 MADISON
53 MADISON
54 MADISON
55 MADISON
56 MADISON
57 MADISON
58 MADISON
59 HAMILTON
60 HAMILTON
61 HAMILTON
62 MADISON
63 MADISON
64 JAY
65 HAMILTON
66 HAMILTON
67 HAMILTON
68 HAMILTON
69 HAMILTON
70 HAMILTON
71 HAMILTON
72 HAMILTON
73 HAMILTON
74 HAMILTON
75 HAMILTON
76 HAMILTON
77 HAMILTON
78 HAMILTON
79 HAMILTON
80 HAMILTON
81 HAMILTON
82 HAMILTON
83 HAMILTON
84 HAMILTON
85 HAMILTO

In [7]:
accepted_essays = []
disputed_essays = []
disregarded_essays = []
disputed_no = [49,50,51,52,53,54,55,56,57,58, 62,63]
author_list_v2 = ['HAMILTON', 'MADISON']

# Papers with joint should be dropped 
for paper in collection_dict: 
    if paper['paper_no'] in disputed_no:
        disputed_essays.append((paper['content'], paper['supposed_author']))
    else:
        if paper['supposed_author'] in author_list_v2:
            accepted_essays.append((paper['content'], paper['supposed_author']))
        else:
            disregarded_essays.append((paper['content'], paper['supposed_author']))

print(len(accepted_essays))
print(len(disputed_essays))
print(len(disregarded_essays))


65
12
8


In [12]:
from collections import Counter

def common_words(text):
    top_n = 50
    ham_vocab = []
    mad_vocab = [] 
    jay_vocab = [] 
    for content, label in text: 
        tokens = [token.text.lower() for token in content]
        if label=='HAMILTON':
            ham_vocab.extend(tokens)
        elif label=='MADISON':
            mad_vocab.extend(tokens)
        elif label=='JAY':
            jay_vocab.extend(tokens)
    ham_top = {w for w, _ in Counter(ham_vocab).most_common(top_n)}
    mad_top = {w for w, _ in Counter(mad_vocab).most_common(top_n)}
    jay_top = {w for w, _ in Counter(jay_vocab).most_common(top_n)}
    return ham_top & mad_top & jay_top
def common_count(text, common_voc):
    common_voc_values = Counter(text)
    counts = {k: common_voc_values[k] for k in common_voc}
    return counts 

In [13]:
def suffix(content):
    suffix_doc = []
    doc_list = [doc for doc, label in content]
    for doc in doc_list:
        for word in doc:
            suffix_doc.append(str(word.suffix_).lower())
    suffix_count = Counter(suffix_doc)
    sorted_count = [suffix for suffix, freq in suffix_count.most_common(50)]
    return sorted_count


def suffix_count(content, suffix_list):
    count_doc = {}
    content_suffix = [str(word.suffix_) for word in content]
    count = Counter(content_suffix)
    for suffix in suffix_list:
        if suffix in count.keys():
            count_doc[suffix] = count.get(suffix)
        else:
            count_doc[suffix] = 0
    return count_doc

In [15]:
default_punct_chars = [
    '!', '.', '?', '։', '؟', '۔', '܀', '܁', '܂', '߹',
    '।', '॥', '၊', '။', '።', '፧', '፨', '᙮', '᜵', '᜶', '᠃', '᠉', '᥄',
    '᥅', '᪨', '᪩', '᪪', '᪫', '᭚', '᭛', '᭞', '᭟', '᰻', '᰼', '᱾', '᱿',
    '‼', '‽', '⁇', '⁈', '⁉', '⸮', '⸼', '꓿', '꘎', '꘏', '꛳', '꛷', '꡶',
    '꡷', '꣎', '꣏', '꤯', '꧈', '꧉', '꩝', '꩞', '꩟', '꫰', '꫱', '꯫', '﹒',
    '﹖', '﹗', '！', '．', '？', '𐩖', '𐩗', '𑁇', '𑁈', '𑂾', '𑂿', '𑃀',
    '𑃁', '𑅁', '𑅂', '𑅃', '𑇅', '𑇆', '𑇍', '𑇞', '𑇟', '𑈸', '𑈹', '𑈻', '𑈼',
    '𑊩', '𑑋', '𑑌', '𑗂', '𑗃', '𑗉', '𑗊', '𑗋', '𑗌', '𑗍', '𑗎', '𑗏', '𑗐',
    '𑗑', '𑗒', '𑗓', '𑗔', '𑗕', '𑗖', '𑗗', '𑙁', '𑙂', '𑜼', '𑜽', '𑜾', '𑩂',
    '𑩃', '𑪛', '𑪜', '𑱁', '𑱂', '𖩮', '𖩯', '𖫵', '𖬷', '𖬸', '𖭄', '𛲟', '𝪈',
    '｡', '。'
]
def punct_count(content):
    punct_count = Counter(punct for punct in content if punct in default_punct_chars)
    return {punct: punct_count.get(punct, 0) for punct in default_punct_chars}

In [17]:
# feature functions:
def word_sentence(text):
    sents_per_text = len([sents for sents in text.sents])
    word_tokens = len([word.text for word in text])
    word_per_sent = float(word_tokens)/float(sents_per_text)
    return word_per_sent, sents_per_text

In [21]:
def type_counter(text, word_per_text): 
    type_count = {"ADJ": 0 , "ADP": 0, "ADV": 0, "AUX": 0, "CCONJ": 0,
    "DET": 0, "INTJ": 0, "NOUN": 0, "NUM": 0, "PART": 0,
    "PRON": 0, "PROPN": 0, "PUNCT": 0, "SCONJ": 0,
    "SYM": 0, "VERB": 0, "X": 0}

    for token in text: 
        if token.pos_ in type_count: 
            type_count[token.pos_] = type_count.get(token.pos_) + 1 
        else: 
            type_count[token.pos_] = 0
    return type_count

In [22]:
def word_count(content):
    word_count = {} 

    for word in content:
        word_lower = word.text.lower()
        if word_lower in word_count.keys(): 
            word_count[word_lower] = word_count.get(word_lower)+1
        else:
            word_count[word_lower] = 1
    return word_count

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import classification_report
from sklearn.base import clone
from sklearn.metrics import confusion_matrix

import numpy as np

content = [nlp(content) for content, label in accepted_essays]
labels = [label for content, label in accepted_essays]
y = np.array(labels)
loo = LeaveOneOut()
y_true, y_pred = [], []
text_clf = DecisionTreeClassifier(random_state=42)
list_importances = []


for train_idx, test_idx in loo.split(content):
    common_voc  = common_words([(content[i], labels[i]) for i in train_idx])
    suffix_list = suffix([(content[i], labels[i]) for i in train_idx])
    noun_chunk = noun_chunking([(content[i], labels[i]) for i in train_idx])
    
    X_train, X_test = [], []
    for i in train_idx:
        word_per_text = len([word for word in content[i]])
        f = document_features(content[i], word_per_text) + corpus_features(content[i], common_voc, suffix_list, word_per_text, noun_chunk)
        X_train.append(f)
    for i in test_idx:
        word_per_text = len(content[i])
        f = document_features(content[i], word_per_text) + corpus_features(content[i], common_voc, suffix_list, word_per_text, noun_chunk)
        X_test.append(f)
    X_train = np.array(X_train)
    X_test  = np.array(X_test)
    y_train = y[train_idx]
    y_test  = y[test_idx]

    model = clone(text_clf)
    model.fit(X_train, y_train)

    list_importances.append({'feature_importances': model.feature_importances_, 'common_voc': common_voc, 'suffix_list': suffix_list, 'noun_chunk': noun_chunk})
    
    predicted = model.predict(X_test)
    y_true.append(y_test[0])
    y_pred.append(predicted[0])

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

NameError: name 'suffix' is not defined

NEW BASELINE

In [24]:
from spacy.lang.en import stop_words as stop_words
stop_words = stop_words.STOP_WORDS
def noun_chunking(content):
    list_doc = [doc for doc, label in content]
    chunk_doc = {} 

    for doc in list_doc: 
        for chunk in doc.noun_chunks:
            chunk_append = chunk.text.lower()
            if len(chunk) > 1:
                if chunk_append in chunk_doc:
                    chunk_doc[chunk_append] = chunk_doc.get(chunk_append)+1
                else:
                    chunk_doc[chunk_append] = 1 
    chunk_list = {chunk for chunk, freq in Counter(chunk_doc).most_common(50)}
    return chunk_list


def chunk_count(content, chunk_list):
    present = Counter(
        chunk.text.lower()
        for chunk in content.noun_chunks
        if len(chunk) > 1
    )
    return {ch: present.get(ch, 0) for ch in chunk_list}

def unique_vocabulary(content, n=50):
    ham_vocab, mad_vocab = [], []

    for doc, label in content:
        for token in doc:
            w = token.text.lower()
            if label == 'HAMILTON':
                ham_vocab.append(w)
            else:
                mad_vocab.append(w)

    ham_top = {w for w, _ in Counter(ham_vocab).most_common(n)}
    mad_top = {w for w, _ in Counter(mad_vocab).most_common(n)}

    only_ham = ham_top - mad_top
    only_mad = mad_top - ham_top 
    # stable order for feature vectors
    unique_vocab = sorted(only_ham | only_mad)
    return unique_vocab


def unique_count(doc, unique_vocab):
    tokens = [token.text.lower() for token in doc]
    count = Counter(t for t in tokens if t in unique_vocab)
    return {w: count.get(w, 0) for w in unique_vocab}

def document_features(doc, word_per_text, punct_list):
    features = []
    word_per_sent, sents_per_text = word_sentence(doc)

    features.append(word_per_sent)
    features.append(sents_per_text)

    pos_counts = type_counter(doc, word_per_text)
    for value in pos_counts.values():
        features.append(value / word_per_text)

    count = word_count(doc)
    for word in sorted(stop_words):
        features.append(count.get(word, 0)/word_per_text)

    for punct, countItem in punct_list.items():
        features.append(countItem/word_per_text)
    return [float(num) for num in features]

def corpus_features(doc, common_voc, suffix_list, word_per_text, unique_list, chunk_list):
    features = []

    count = word_count(doc)
    for word in common_voc:
        if word not in default_punct_chars:
            features.append(count.get(word, 0)/word_per_text)

    count_doc = suffix_count(doc, suffix_list)        
    for suffix in count_doc.keys():
        features.append(count_doc.get(suffix)/word_per_text)

    unique_doc = unique_count(doc, unique_list)
    for unique in unique_list:  
        features.append(unique_doc.get(unique, 0)/word_per_text) 

    chunk_doc = chunk_count(doc, chunk_list)
    for chunk in chunk_list:  
        features.append(chunk_doc.get(chunk, 0)) 

    return [float(num) for num in features]


content = [nlp(content) for content, label in accepted_essays]
labels = [label for content, label in accepted_essays]
y = np.array(labels)
loo = LeaveOneOut()
y_true, y_pred = [], []
text_clf = DecisionTreeClassifier(random_state=42)
list_importances = []


for train_idx, test_idx in loo.split(content):
    common_voc  = common_words([(content[i], labels[i]) for i in train_idx])
    suffix_list = suffix([(content[i], labels[i]) for i in train_idx])
    punct_list = punct_count([(content[i], labels[i]) for i in train_idx])
    unique_vocab_list = unique_vocabulary([(content[i], labels[i]) for i in train_idx])
    noun_chunk = noun_chunking([(content[i], labels[i]) for i in train_idx])

    X_train, X_test = [], []
    for i in train_idx:
        word_per_text = len([word for word in content[i]])
        f = document_features(content[i], word_per_text, punct_list) + corpus_features(content[i], common_voc, suffix_list, word_per_text, unique_vocab_list, noun_chunk)
        X_train.append(f)
    for i in test_idx:
        word_per_text = len(content[i])
        f = document_features(content[i], word_per_text, punct_list) + corpus_features(content[i], common_voc, suffix_list, word_per_text, unique_vocab_list, noun_chunk)
        X_test.append(f)
    X_train = np.array(X_train)
    X_test  = np.array(X_test)
    y_train = y[train_idx]
    y_test  = y[test_idx]

    model = clone(text_clf)
    model.fit(X_train, y_train)
    
    predicted = model.predict(X_test)
    y_true.append(y_test[0])
    y_pred.append(predicted[0])

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

    HAMILTON       0.96      0.98      0.97        51
     MADISON       0.92      0.86      0.89        14

    accuracy                           0.95        65
   macro avg       0.94      0.92      0.93        65
weighted avg       0.95      0.95      0.95        65

[[50  1]
 [ 2 12]]


Experiment

In [ ]:
def n_gram(text, n):
    if not isinstance(text, str):
        text = text.text
    text = text.lower()
    return [text[i:i+n] for i in range(len(text)-n+1)]

def n_vocab(content, n):
    counts = Counter()
    for doc,label in content: 
        counts.update(n_gram(doc, n))
    return [ng for ng,_ in counts.most_common(500)]

def n_vocab_count(paper, n, set_vocab):
    vocab_count = Counter(n_gram(paper, n))
    return {n: vocab_count.get(n, 0) for n in set_vocab}

In [ ]:
def document_features(doc, word_per_text):
    features = []
    word_per_sent, sents_per_text = word_sentence(doc)

    features.append(word_per_sent)
    features.append(sents_per_text)

    pos_counts = type_counter(doc, word_per_text)
    for value in pos_counts.values():
        features.append(value /float(word_per_text))

    count = word_count(doc)
    for word in sorted(stop_words):
        features.append(count.get(word, 0)/float(word_per_text) )
    return [float(num) for num in features]

def corpus_features(doc, common_voc, suffix_list, word_per_text, chunk_list, n_list, n_per_text):
    features = []

    count = word_count(doc)
    for word in common_voc:
        features.append(count.get(word, 0)/float(word_per_text) )

    count_doc = suffix_count(doc, suffix_list)        
    for suffix in count_doc.keys():
        features.append(count_doc.get(suffix)/float(word_per_text) )

    chunk_doc = chunk_count(doc, chunk_list)
    for chunk in chunk_list:  
        features.append(chunk_doc.get(chunk, 0)/float(word_per_text) ) 

    n_doc = n_vocab_count(doc, 3, n_list)
    for n in n_doc:
        features.append(n_doc.get(n, 0)/float(n_per_text))
    return [float(num)for num in features]

content = [nlp(content) for content, label in accepted_essays]
labels = [label for content, label in accepted_essays]
y = np.array(labels)
loo = LeaveOneOut()
y_true, y_pred = [], []
text_clf = DecisionTreeClassifier(random_state=42)

for train_idx, test_idx in loo.split(content):
    n=3
    common_voc  = common_words([(content[i], labels[i]) for i in train_idx])
    suffix_list = suffix([(content[i], labels[i]) for i in train_idx])
    noun_chunk = noun_chunking([(content[i], labels[i]) for i in train_idx])
    n_list = n_vocab([(content[i], labels[i]) for i in train_idx], n)
    
    X_train, X_test = [], []
    for i in train_idx:
        doc = content[i]
        word_per_text = max(len(doc), 1)
        n_per_text = max(len(doc.text) - n + 1, 1)
        f = document_features(content[i], word_per_text) + corpus_features(content[i], common_voc, suffix_list, word_per_text, noun_chunk, n_list, n_per_text)
        X_train.append(f)
    for i in test_idx:
        doc = content[i]
        word_per_text = max(len(doc), 1)
        n_per_text = max(len(doc.text) - n + 1, 1)
        f = document_features(content[i], word_per_text) + corpus_features(content[i], common_voc, suffix_list, word_per_text, noun_chunk, n_list, n_per_text)
        X_test.append(f)
    X_train = np.array(X_train)
    X_test  = np.array(X_test)
    y_train = y[train_idx]
    y_test  = y[test_idx]

    model = clone(text_clf)
    model.fit(X_train, y_train)
    
    predicted = model.predict(X_test)
    y_true.append(y_test[0])
    y_pred.append(predicted[0])

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

    HAMILTON       0.98      0.98      0.98        51
         JAY       0.80      0.80      0.80         5
     MADISON       0.86      0.86      0.86        14

    accuracy                           0.94        70
   macro avg       0.88      0.88      0.88        70
weighted avg       0.94      0.94      0.94        70

[[50  0  1]
 [ 0  4  1]
 [ 1  1 12]]


In [ ]:
from spacy.lang.en import stop_words as stop_words
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler


stop_words = stop_words.STOP_WORDS
def noun_chunking(content):
    list_doc = [doc for doc, label in content]
    chunk_doc = {} 

    for doc in list_doc: 
        for chunk in doc.noun_chunks:
            chunk_append = chunk.text.lower()
            if len(chunk) > 1:
                if chunk_append in chunk_doc:
                    chunk_doc[chunk_append] = chunk_doc.get(chunk_append)+1
                else:
                    chunk_doc[chunk_append] = 1 
    chunk_list = {chunk for chunk, freq in Counter(chunk_doc).most_common(50)}
    return chunk_list


def chunk_count(content, chunk_list):
    present = Counter(
        chunk.text.lower()
        for chunk in content.noun_chunks
        if len(chunk) > 1
    )
    return {ch: present.get(ch, 0) for ch in chunk_list}

def unique_vocabulary(content, n=50):
    ham_vocab, mad_vocab, jay_vocab = [], [], []

    for doc, label in content:
        for token in doc:
            w = token.text.lower()
            if label == 'HAMILTON':
                ham_vocab.append(w)
            elif label == 'MADISON':
                mad_vocab.append(w)
            else:
                jay_vocab.append(w)

    ham_top = {w for w, _ in Counter(ham_vocab).most_common(n)}
    mad_top = {w for w, _ in Counter(mad_vocab).most_common(n)}
    jay_top = {w for w, _ in Counter(jay_vocab).most_common(n)}

    only_ham = ham_top - mad_top - jay_top
    only_mad = mad_top - ham_top - jay_top
    only_jay = jay_top - ham_top - mad_top

    # stable order for feature vectors
    unique_vocab = sorted(only_ham | only_mad | only_jay)
    return unique_vocab


def unique_count(doc, unique_vocab):
    tokens = [token.text.lower() for token in doc]
    count = Counter(t for t in tokens if t in unique_vocab)
    return {w: count.get(w, 0) for w in unique_vocab}

def document_features(doc, word_per_text, punct_list):
    features = []
    word_per_sent, sents_per_text = word_sentence(doc)

    features.append(word_per_sent)
    features.append(sents_per_text)

    pos_counts = type_counter(doc, word_per_text)
    for value in pos_counts.values():
        features.append(value / word_per_text)

    count = word_count(doc)
    for word in sorted(stop_words):
        features.append(count.get(word, 0)/word_per_text)

    for punct, countItem in punct_list.items():
        features.append(countItem/word_per_text)
    return [float(num) for num in features]

def corpus_features(doc, common_voc, suffix_list, word_per_text, unique_list, chunk_list):
    features = []

    count = word_count(doc)
    for word in common_voc:
        if word not in default_punct_chars:
            features.append(count.get(word, 0)/word_per_text)

    count_doc = suffix_count(doc, suffix_list)        
    for suffix in count_doc.keys():
        features.append(count_doc.get(suffix)/word_per_text)

    unique_doc = unique_count(doc, unique_list)
    for unique in unique_list:  
        features.append(unique_doc.get(unique, 0)/word_per_text) 

    chunk_doc = chunk_count(doc, chunk_list)
    for chunk in chunk_list:  
        features.append(chunk_doc.get(chunk, 0)) 

    return [float(num) for num in features]

# ... your existing helper functions stay the same ...

content = [nlp(text) for text, label in accepted_essays]
labels = [label for text, label in accepted_essays]
y = np.array(labels)

loo = LeaveOneOut()
y_true, y_pred = [], []
text_clf = SVC(kernel='linear', random_state=42)

for train_idx, test_idx in loo.split(content):
    # --- feature extraction (same as before) ---
    common_voc       = common_words([(content[i], labels[i]) for i in train_idx])
    suffix_list      = suffix([(content[i], labels[i]) for i in train_idx])
    punct_list       = punct_count([(content[i], labels[i]) for i in train_idx])
    unique_vocab_list = unique_vocabulary([(content[i], labels[i]) for i in train_idx])
    noun_chunk       = noun_chunking([(content[i], labels[i]) for i in train_idx])

    X_train, X_test = [], []
    for i in train_idx:
        word_per_text = len([word for word in content[i]])
        f = (document_features(content[i], word_per_text, punct_list) +
             corpus_features(content[i], common_voc, suffix_list,
                             word_per_text, unique_vocab_list, noun_chunk))
        X_train.append(f)

    for i in test_idx:
        word_per_text = len(content[i])
        f = (document_features(content[i], word_per_text, punct_list) +
             corpus_features(content[i], common_voc, suffix_list,
                             word_per_text, unique_vocab_list, noun_chunk))
        X_test.append(f)

    X_train = np.array(X_train, dtype=float)
    X_test  = np.array(X_test,  dtype=float)
    y_train = y[train_idx]
    y_test  = y[test_idx]

    # --- scaling (fit on train only) ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # --- train & predict ---
    model = clone(text_clf)
    model.fit(X_train_scaled, y_train)

    predicted = model.predict(X_test_scaled)
    y_true.append(y_test[0])
    y_pred.append(predicted[0])

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

    HAMILTON       0.96      1.00      0.98        51
         JAY       1.00      0.60      0.75         5
     MADISON       0.86      0.86      0.86        14

    accuracy                           0.94        70
   macro avg       0.94      0.82      0.86        70
weighted avg       0.94      0.94      0.94        70

[[51  0  0]
 [ 0  3  2]
 [ 2  0 12]]


In [25]:
# Accepted: known labels
accepted_docs   = [nlp(text) for text, label in accepted_essays]
accepted_labels = [label for text, label in accepted_essays]

# Disputed: texts only (no labels used for training)
disputed_docs = [nlp(text) for text, label in disputed_essays]
# or if you only have raw strings:
# disputed_docs = [nlp(text) for text in disputed_texts]

In [26]:
def suffix(docs, n=50):
    suffix_doc = []
    for doc in docs:
        for word in doc:
            suffix_doc.append(str(word.suffix_).lower())
    return [s for s, _ in Counter(suffix_doc).most_common(n)]


def suffix_count(doc, suffix_list):
    count = Counter(str(word.suffix_).lower() for word in doc)
    return {suf: count.get(suf, 0) for suf in suffix_list}


def noun_chunking(docs, n=50):
    chunk_doc = Counter()
    for doc in docs:
        for chunk in doc.noun_chunks:
            if len(chunk) > 1:
                chunk_doc[chunk.text.lower()] += 1
    return [c for c, _ in chunk_doc.most_common(n)]

def chunk_count(doc, chunk_list):
    present = Counter(
        chunk.text.lower()
        for chunk in doc.noun_chunks
        if len(chunk) > 1
    )
    return {ch: present.get(ch, 0) for ch in chunk_list}

In [27]:
def document_features(doc, word_per_text):
    features = []
    word_per_sent, sents_per_text = word_sentence(doc)

    features.append(word_per_sent)
    features.append(sents_per_text)

    pos_counts = type_counter(doc, word_per_text)
    for value in pos_counts.values():
        features.append(value /float(word_per_text))

    count = word_count(doc)
    for word in sorted(stop_words):
        features.append(count.get(word, 0)/float(word_per_text) )
    return [float(num) for num in features]

def corpus_features(doc, common_voc, suffix_list, word_per_text, chunk_list, n_list, n_per_text):
    features = []

    count = word_count(doc)
    for word in common_voc:
        features.append(count.get(word, 0)/float(word_per_text) )

    count_doc = suffix_count(doc, suffix_list)        
    for suffix in count_doc.keys():
        features.append(count_doc.get(suffix)/float(word_per_text) )

    chunk_doc = chunk_count(doc, chunk_list)
    for chunk in chunk_list:  
        features.append(chunk_doc.get(chunk, 0)/float(word_per_text) ) 

    n_doc = n_vocab_count(doc, 3, n_list)
    for n in n_doc:
        features.append(n_doc.get(n, 0)/float(n_per_text))
    return [float(num)for num in features]

In [28]:
suffix_list = suffix(accepted_docs)                 # list of Docs
chunk_list  = noun_chunking(accepted_docs)          # list of Docs
common_voc  = common_words(list(zip(accepted_docs, accepted_labels)))
# unique_vocab = unique_vocabulary(list(zip(accepted_docs, accepted_labels)))

def featurize(doc, common_voc, suffix_list, chunk_list):
    wpt = len(doc)
    return (
        document_features(doc, wpt)
        + corpus_features(doc, common_voc, suffix_list, wpt, chunk_list)
    )

In [29]:
from sklearn.tree import DecisionTreeClassifier

In [30]:
X_accepted = np.array([
    featurize(doc, common_voc, suffix_list, chunk_list)
    for doc in accepted_docs
])

# map labels to ints (same mapping as in LOOCV)
label_to_id = {"HAMILTON": 0, "MADISON": 1, "JAY": 2}
id_to_label = {0: "HAMILTON", 1: "MADISON", 2: "JAY"}

y_accepted = np.array([label_to_id[lab] for lab in accepted_labels])

final_model = DecisionTreeClassifier(random_state=42)
final_model.fit(X_accepted, y_accepted)

TypeError: corpus_features() missing 2 required positional arguments: 'n_list' and 'n_per_text'

In [ ]:
X_disputed = np.array([
    featurize(doc, common_voc, suffix_list, chunk_list)
    for doc in disputed_docs
])

pred_ids = final_model.predict(X_disputed)
probas   = final_model.predict_proba(X_disputed)  

for i, (pid, proba) in enumerate(zip(pred_ids, probas)):
    author = id_to_label[pid]
    print(f"Disputed {i+1:2d}: {author:8s}  "
          f"(H={proba[0]:.2f}  M={proba[1]:.2f}  J={proba[2]:.2f})")

Disputed  1: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed  2: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed  3: HAMILTON  (H=1.00  M=0.00  J=0.00)
Disputed  4: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed  5: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed  6: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed  7: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed  8: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed  9: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed 10: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed 11: MADISON   (H=0.00  M=1.00  J=0.00)
Disputed 12: MADISON   (H=0.00  M=1.00  J=0.00)
